In [1]:
!pip install -q uv
!uv pip install --system \
  "transformers==5.4.0" accelerate bitsandbytes peft trl datasets \
  soundfile librosa mutagen openpyxl datacollective jiwer "unsloth==2026.8.18" \
  "unsloth-zoo==2026.8.12" evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 66.3 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 128 packages in 997ms                                       
Prepared 21 packages in 3.09s                                            
Uninstalled 5 packages in 746ms
Installed 21 packages in 129ms                              
 + bitsandbytes==0.50.1
 + cut-cross-entropy==25.1.1
 + datacollective==0.5.7
 - datasets==5.0.0
 + datasets==4.3.0
 - dill==0.4.1
 + dill==0.4.0
 + evaluate==0.4.6
 + fox-progress-bar==0.1.3
 + hf-transfer==0.1.9
 + jiwer==4.0.0
 + msgspec==0.21.1
 + mutagen==1.48.1
 + rapidfuzz==3.14.5
 - requests==2.32.4
 + requests==2.34.2
 + structlog==26.1.0
 - torchao==0.10.0
 + torchao==0.18.0
 - transformers==5.0.0
 + transformers==5.4.0
 + trl==0.24.0
 + tyro==1.0.15
 + unsloth==2026.8.18
 + unsloth-zoo==2026.8.12
 + xformers==0.0.35


In [2]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import os
import csv
import glob
import random
import pandas as pd
import torch
from mutagen.mp3 import MP3
from datacollective import download_dataset
import tarfile
from huggingface_hub import login, hf_hub_download
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import soundfile as sf
import librosa
import evaluate

WHISPER_FT_REPO = "amirsz8203/whisper-small-fa-finetuned"
WHISPER_BASE = "openai/whisper-small"
GEMMA_FT_REPO = "amirsz8203/gemma3-fa-whisper-refinement-lora-v13"
GEMMA_BASE = "unsloth/gemma-3-4b-it"
LANGUAGE = "persian"
TASK = "transcribe"
SAMPLE_RATE = 16000
SEED = 999  # seed جدید و متفاوت، تا نمونه‌ها کاملاً تازه باشن
N_TEST = 100

CV_EXTRACT_DIR = "/tmp/common_voice_fa_extracted"
LLM_DATASET_PATH = "/kaggle/input/datasets/amirsafarzadeh8203/raw-pairs-v10k-cleaned/raw_pairs_v10k_cleaned.csv"  # <-- مسیر واقعی رو بذار

random.seed(SEED)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    MDC_API_KEY = UserSecretsClient().get_secret("MDC_API_KEY")
except Exception:
    HF_TOKEN = "خودتان جایگذاری کنید"
    MDC_API_KEY = "خودتان جایگذاری کنید"

os.environ["MDC_API_KEY"] = MDC_API_KEY
login(token=HF_TOKEN)

n_gpu = torch.cuda.device_count()
print("تعداد GPU:", n_gpu)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
تعداد GPU: 2


In [3]:
CV_DATASET_ID = "cmqinhw5100v8nr07gyg5gi4v"


cv_archive_path = str(download_dataset(CV_DATASET_ID))

if os.path.isfile(cv_archive_path):
    if not os.path.isdir(CV_EXTRACT_DIR) or not os.listdir(CV_EXTRACT_DIR):
        os.makedirs(CV_EXTRACT_DIR, exist_ok=True)
        print("در حال extract کردن... (چند دقیقه طول می‌کشه)")
        with tarfile.open(cv_archive_path, "r:gz") as tar:
            tar.extractall(path=CV_EXTRACT_DIR)
    cv_root = CV_EXTRACT_DIR
elif os.path.isdir(cv_archive_path):
    cv_root = cv_archive_path
else:
    raise RuntimeError(f"مسیر برگشتی نه فایله نه پوشه: {cv_archive_path}")

tsv_candidates = (
    glob.glob(os.path.join(cv_root, "**", "validated.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "train.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "*.tsv"), recursive=True)
)
cv_tsv_path = tsv_candidates[0]
mp3_candidates = glob.glob(os.path.join(cv_root, "**", "*.mp3"), recursive=True)
clips_dir = os.path.dirname(mp3_candidates[0])

cv_df = pd.read_csv(cv_tsv_path, sep="\t", quoting=csv.QUOTE_NONE)
cv_df["full_path"] = cv_df["path"].apply(lambda p: os.path.join(clips_dir, p))
print("تعداد کل ردیف‌های متادیتا:", len(cv_df))

█████████████████████████████████████████████████🦊 100.0% (10.5 GB/10.5 GB) Average: 75.9 MB/s Total time: 02:21
در حال extract کردن... (چند دقیقه طول می‌کشه)


/tmp/ipykernel_58/304933276.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=CV_EXTRACT_DIR)
/tmp/ipykernel_58/304933276.py:29: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  cv_df = pd.read_csv(cv_tsv_path, sep="\t", quoting=csv.QUOTE_NONE)


تعداد کل ردیف‌های متادیتا: 341657


In [4]:
# کلیپ‌هایی که Whisper دیده
whisper_used_file = hf_hub_download(repo_id=WHISPER_FT_REPO, filename="used_clips.txt")
with open(whisper_used_file) as f:
    whisper_used_paths = set(line.strip() for line in f if line.strip())

# کلیپ‌هایی که Gemma دیده
llm_dataset_df = pd.read_csv(LLM_DATASET_PATH)
gemma_used_paths = set(llm_dataset_df["path"])

# فقط کلیپ‌هایی که هیچ‌کدوم ندیدن
fully_unused_df = cv_df[
    ~cv_df["full_path"].isin(whisper_used_paths) & ~cv_df["full_path"].isin(gemma_used_paths)
]
fully_unused_df = fully_unused_df[fully_unused_df["sentence"].astype(str).str.len().between(10, 300)]

test_df = fully_unused_df.sample(n=N_TEST, random_state=SEED).reset_index(drop=True)
print(f"تعداد نمونه‌های تست (کاملاً تازه برای هر دو مدل): {len(test_df)}")

used_clips.txt: 0.00B [00:00, ?B/s]

تعداد نمونه‌های تست (کاملاً تازه برای هر دو مدل): 100


In [5]:
# processor_base = WhisperProcessor.from_pretrained(WHISPER_BASE, language=LANGUAGE, task=TASK)
# whisper_base = WhisperForConditionalGeneration.from_pretrained(WHISPER_BASE).to("cuda:0")
# whisper_base.generation_config.language = LANGUAGE
# whisper_base.generation_config.task = TASK
# whisper_base.generation_config.forced_decoder_ids = None
# whisper_base.eval()

processor_ft = WhisperProcessor.from_pretrained(WHISPER_FT_REPO, language=LANGUAGE, task=TASK)
whisper_ft = WhisperForConditionalGeneration.from_pretrained(WHISPER_FT_REPO).to("cuda:0")
whisper_ft.generation_config.language = LANGUAGE
whisper_ft.generation_config.task = TASK
whisper_ft.generation_config.forced_decoder_ids = None
whisper_ft.eval()

def transcribe(path, model, processor):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    input_features = processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE).input_features[0]
    input_features = torch.tensor(input_features).unsqueeze(0).to(model.device)
    with torch.no_grad():
        pred_ids = model.generate(input_features)
    return processor.batch_decode(pred_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()



processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [6]:
# gemma_base_model, gemma_base_tokenizer = FastModel.from_pretrained(
#     model_name=GEMMA_BASE, max_seq_length=1024, load_in_4bit=True,
# )
# gemma_base_tokenizer = get_chat_template(gemma_base_tokenizer, chat_template="gemma3")
# FastModel.for_inference(gemma_base_model)

gemma_ft_model, gemma_ft_tokenizer = FastModel.from_pretrained(
    model_name=GEMMA_FT_REPO, max_seq_length=1024, load_in_4bit=True,
)
gemma_ft_tokenizer = get_chat_template(gemma_ft_tokenizer, chat_template="gemma3")
FastModel.for_inference(gemma_ft_model)

print("هر دو مدل Gemma لود شدن.")

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.8.18: Fast Gemma3 patching. Transformers: 5.4.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

هر دو مدل Gemma لود شدن.


In [7]:
SYSTEM_PROMPT_FT = """متن زیر خروجی خام یک سیستم تشخیص گفتار (ASR) فارسیه.
وظیفه‌ی تو اصلاح نیم‌فاصله، علائم نگارشی، و تصحیح کلماتیه که به‌اشتباه توسط
سیستم تشخیص گفتار جایگزین شده‌اند (مثلاً به‌خاطر شباهت آوایی، مثل "گرمز" به‌جای "قرمز").

قوانین سخت‌گیرانه که هرگز نباید نقض بشن:
- تعداد کلمات نباید تغییر کند؛ فقط جایگزینیِ یک کلمه با کلمه‌ی دیگر مجاز است،
  نه حذف یا اضافه‌کردن کلمه.
- اگر مطمئن نیستی یک کلمه اشتباه است یا نه، همان‌طور که هست نگهش دار.
- اگر متن از قبل کاملاً درست است، دقیقاً همان را بدون هیچ تغییری برگردان.
- خروجی فقط باید خودِ متن نهایی باشد، بدون توضیح، بدون علامت نقل‌قول اضافه."""


def punctuation_count(text):
    return sum(text.count(c) for c in "؟?!.")


def refine_text(raw_text, model, tokenizer):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_FT}]},
        {"role": "user", "content": [{"type": "text", "text": raw_text}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=256, do_sample=False)
    result = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

    if len(result.split()) != len(raw_text.split()):
        return raw_text
    if punctuation_count(result) < punctuation_count(raw_text):
        return raw_text
    return result

In [8]:
records = []
for i, row in test_df.iterrows():
    path = row["full_path"]
    actual = str(row["sentence"])
    # raw = transcribe(path, whisper_base, processor_base)
    raw = transcribe(path, whisper_ft, processor_ft)
    # refined = refine_text(raw, gemma_base_model, gemma_base_tokenizer)
    refined = refine_text(raw, gemma_ft_model, gemma_ft_tokenizer)
    records.append({"متن واقعی": actual, "متن Whisper": raw, "متن LLM": refined})
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(test_df)} پردازش شد")

df_out = pd.DataFrame(records)
# out_path = "/kaggle/working/comparison_1_no_finetuning.xlsx"
# out_path = "/kaggle/working/comparison_2_whisper_finetuned_only.xlsx"
# out_path = "/kaggle/working/comparison_3_gemma_finetuned_only.xlsx"
out_path = "/kaggle/working/comparison_4_both_finetuned.xlsx"
df_out.to_excel(out_path, index=False, engine="openpyxl")
print("ذخیره شد:", out_path)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

10/100 پردازش شد
20/100 پردازش شد
30/100 پردازش شد
40/100 پردازش شد
50/100 پردازش شد
60/100 پردازش شد
70/100 پردازش شد
80/100 پردازش شد
90/100 پردازش شد
100/100 پردازش شد
ذخیره شد: /kaggle/working/comparison_4_both_finetuned.xlsx


In [17]:
import pandas as pd
import evaluate

wer_metric = evaluate.load("wer")

excel_files = {
    "1_no_finetuning": "/kaggle/input/datasets/amirsafarzadeh8203/no-finetuning/comparison_1_no_finetuning.xlsx",
    "2_whisper_finetuned_only": "/kaggle/input/datasets/amirsafarzadeh8203/whisper-finetuned-only/comparison_2_whisper_finetuned_only.xlsx",
    "3_gemma_finetuned_only": "/kaggle/input/datasets/amirsafarzadeh8203/gemma-finetuned-only/comparison_3_gemma_finetuned_only.xlsx",
    "4_both_finetuned": "/kaggle/input/datasets/amirsafarzadeh8203/both-finetuned/comparison_4_both_finetuned.xlsx",
}

summary = []
for name, path in excel_files.items():
    df = pd.read_excel(path)

    wer_whisper = 100 * wer_metric.compute(
        predictions=df["متن Whisper"].astype(str).tolist(),
        references=df["متن واقعی"].astype(str).tolist(),
    )
    wer_llm = 100 * wer_metric.compute(
        predictions=df["متن LLM"].astype(str).tolist(),
        references=df["متن واقعی"].astype(str).tolist(),
    )

    summary.append({
        "scenario": name,
        "WER Whisper": round(wer_whisper, 2),
        "WER after LLM": round(wer_llm, 2),
        "change": round(wer_llm - wer_whisper, 2),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

                scenario  WER Whisper  WER after LLM  change
         1_no_finetuning       108.33         107.61   -0.72
2_whisper_finetuned_only        26.87          28.74    1.87
  3_gemma_finetuned_only       108.33         107.18   -1.15
        4_both_finetuned        26.87          28.30    1.44
